use the the repo list from step 1.3 to lookup for metadata


In [ ]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv
from time import sleep

# === Load GitHub tokens ===
load_dotenv("All_Tokens.env")
tokens = [os.getenv(f"GITHUB_TOKEN_{i}") for i in range(1, 7)]
tokens = [t for t in tokens if t]
if not tokens:
    raise ValueError("No GitHub tokens found in All_Tokens.env")

token_index = 0

def get_headers():
    global token_index
    token = tokens[token_index % len(tokens)]
    token_index += 1
    return {"Authorization": f"token {token}", "Accept": "application/vnd.github+json"}

# === Read input file ===
input_path = r"C:/GitHub/Android_Testing/Repo List - Stepwise - With Removal Label.csv"
df = pd.read_csv(input_path)
html_urls = df['html_urls'].dropna().unique()

# === Extract metadata from GitHub REST API ===
def get_repo_metadata(url):
    try:
        parts = url.strip().split("github.com/")[-1].strip("/").split("/")
        if len(parts) < 2:
            return None
        owner, repo = parts[0], parts[1]
        api_url = f"https://api.github.com/repos/{owner}/{repo}"
        response = requests.get(api_url, headers=get_headers())
        if response.status_code == 200:
            data = response.json()
            return {
                "html_url": url,
                "id": data.get("id"),
                "name": data.get("name"),
                "full_name": data.get("full_name"),
                "owner": data.get("owner", {}).get("login"),
                "private": data.get("private"),
                #"description": data.get("description"),
                "fork": data.get("fork"),
                "created_at": data.get("created_at"),
                "updated_at": data.get("updated_at"),
                "pushed_at": data.get("pushed_at"),
                "homepage": data.get("homepage"),
                "size": data.get("size"),
                "stargazers_count": data.get("stargazers_count"),
                "watchers_count": data.get("watchers_count"),
                "language": data.get("language"),
                "forks_count": data.get("forks_count"),
                "open_issues_count": data.get("open_issues_count"),
                "license": data.get("license", {}).get("name") if data.get("license") else None,
                "topics": ", ".join(data.get("topics", [])),
                "visibility": data.get("visibility"),
                "default_branch": data.get("default_branch"),
                "has_issues": data.get("has_issues"),
                "has_projects": data.get("has_projects"),
                "has_downloads": data.get("has_downloads"),
                "has_wiki": data.get("has_wiki"),
                "has_pages": data.get("has_pages"),
                "archived": data.get("archived"),
                "disabled": data.get("disabled"),
                "allow_forking": data.get("allow_forking"),
                "is_template": data.get("is_template"),
                "web_commit_signoff_required": data.get("web_commit_signoff_required")
            }
        else:
            print(f"[{response.status_code}] Skipped: {url}")
            return None
    except Exception as e:
        print(f"[ERROR] {url}: {e}")
        return None

# === Process all URLs ===
results = []
total = len(html_urls)
for idx, url in enumerate(html_urls, 1):
    print(f"[{idx}/{total}] Fetching metadata for: {url}")
    meta = get_repo_metadata(url)
    if meta:
        results.append(meta)
    sleep(1.2)  # Respect rate limits

# === Save output ===
output_df = pd.DataFrame(results)
output_file = "/mnt/data/Repo_Metadata_Full_Output.csv"
output_df.to_csv(output_file, index=False)
print(f"\n✅ All metadata saved to: {output_file}")


[1/25812] Fetching metadata for: https://github.com/0015/ThatProject
[2/25812] Fetching metadata for: https://github.com/008chen/InterpolatorShow
[3/25812] Fetching metadata for: https://github.com/00ec454/Ask
[4/25812] Fetching metadata for: https://github.com/00ec454/pop
[5/25812] Fetching metadata for: https://github.com/00-Evan/shattered-pixel-dungeon
[6/25812] Fetching metadata for: https://github.com/06peng/FrescoDemo
[7/25812] Fetching metadata for: https://github.com/08carmelo/android-keeplive
[8/25812] Fetching metadata for: https://github.com/0maru/twitter_login
[9/25812] Fetching metadata for: https://github.com/0niel/university-app
[10/25812] Fetching metadata for: https://github.com/0ranko0P/AutoDark
[11/25812] Fetching metadata for: https://github.com/0x4f53/Wristkey
[12/25812] Fetching metadata for: https://github.com/0x5e/RubiksCubeSolver
[13/25812] Fetching metadata for: https://github.com/0x7c13/Pal3.Unity
[14/25812] Fetching metadata for: https://github.com/0xbad1d3a